# Chapter 16b — Flash Attention from Scratch

> Course: **llm.c — Zero to Hero**, Chapter 16b (deep dive after Chapter 16).
> Builds on **Chapter 6** (CPU attention), **Chapters 12-13** (warp + block reductions, shared memory), and **Chapter 16** (GPU attention, which *teased* Flash Attention via cuDNN but never opened the box).

Chapter 16 ended on a cliffhanger: the production path calls cuDNN's `attention_forward_cudnn`, a Flash Attention kernel, and we treated it as magic. This chapter opens the box. By the end you will have **written a tiled Flash Attention forward kernel in CUDA by hand**, verified it against a reference, and implemented the **backward pass** — the part most tutorials skip.

Flash Attention is not a different attention. It computes the **exact** same `softmax(QKᵀ)V` — bit-for-bit, no approximation. What changes is *how it moves memory*. That single idea (be **IO-aware**) is why it is faster and why it fits sequences that standard attention cannot.

### Learning objectives

By the end of this chapter you will be able to:

- Explain **IO-awareness**: why attention is bound by **HBM traffic**, not FLOPs, and why the `(B, NH, T, T)` scores matrix is the enemy.
- Derive the **online softmax** recurrence and prove it equals the naive softmax in one pass with `O(1)` state per row.
- Read **Algorithm 1** of the FlashAttention paper and map every line to tiling over Q/K/V blocks.
- **Write and run** a tiled Flash Attention forward CUDA kernel from scratch and verify it against a CPU reference.
- Explain the **backward pass**: recomputation (selective checkpointing) and the `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)` trick that collapses the softmax Jacobian — and verify it numerically.
- Contrast **FlashAttention-1 vs FlashAttention-2** (loop order, parallelism, warp work-partitioning).

### Sources grounding this chapter

- Dao, Fu, Ermon, Rudra, Ré — [*FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*](https://arxiv.org/abs/2205.14135) (2022). Algorithm 1, Theorems 1-2, Figure 2.
- Dao — [*FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*](https://arxiv.org/abs/2307.08691) (2023).
- [`tspeterkim/flash-attention-minimal`](https://github.com/tspeterkim/flash-attention-minimal) and [`66RING/tiny-flash-attention`](https://github.com/66RING/tiny-flash-attention) — the minimal CUDA kernels our code mirrors.
- [`sonnyli/flash_attention_from_scratch`](https://github.com/sonnyli/flash_attention_from_scratch) — derivations and a performance-tuned implementation.
- Stanford **CS224n (2024), Lecture 18 — Deployment & Efficiency** — the step-by-step "tiling → rescaling → recomputation" build-up this chapter's §2c and §6 mirror.


## 1. The Problem — Attention Is Memory-Bound, Not Compute-Bound

Recall standard attention for one head with sequence length `T` and head dim `d`:

```
S = Q Kᵀ · scale     # (T, T)   <- the scores matrix
P = softmax(S)        # (T, T)   <- row-wise softmax
O = P V               # (T, d)
```

The naive GPU implementation (Chapter 16) does this as three steps, each a separate kernel that **reads its inputs from HBM and writes its output back to HBM**:

1. `S = QKᵀ` — write `T×T` floats to HBM.
2. `P = softmax(S)` — read `T×T`, write `T×T`.
3. `O = PV` — read `T×T`.

The `(T, T)` matrix is the villain. It scales **quadratically** in sequence length, and every byte of it makes a round trip to slow off-chip memory. Count the trips: the scores matrix is **written** (step 1), **read** for the softmax (step 2), **written** back (step 2), and **read** again for `PV` (step 3) — roughly **four `T×T` round trips**. For `T = 4096` in fp16 that one matrix is `4096² × 2 B ≈ 34 MB`, so attention moves **~134 MB through HBM per head** just shuttling scores — while the actual arithmetic finishes in microseconds. The kernel sits idle waiting on memory.


![GPU memory hierarchy](course/figures/fig_16b_memory_hierarchy.png)

On-chip **SRAM** (shared memory) is ~10x faster than **HBM** but thousands of times smaller. A modern GPU can do far more arithmetic per second than it can feed from HBM, so any kernel that touches a lot of HBM per FLOP is **memory-bound** — it sits idle waiting for data. Attention's softmax and the `(T,T)` traffic make it exactly that.

The fix is not "do less math" — Flash Attention actually does **more** FLOPs (it recomputes things). The fix is "**touch HBM less**." That trade — more compute to avoid memory traffic — is the whole game.


![Scores memory blowup](course/figures/fig_16b_materialization.png)

The quadratic growth is brutal. For **one** head's `T×T` scores in fp32, watch the size explode as `T` doubles:

| Sequence length `T` | `T×T` scores (fp32) |
|---|---|
| 1,024 | 4 MB |
| 4,096 | 67 MB |
| 16,384 | 1 GB |
| 32,768 | 4 GB |

And that is *one head*. For GPT-2-small dimensions (`B=8, NH=12`) at `T=16384` in fp32, the scores tensor across all batches and heads is **hundreds of GB** — it does not fit in any GPU. Flash Attention never allocates it: its only persistent extra state is two scalars per row, the running `(m, ℓ)`, which is `O(T)`, the green line hugging the bottom.

> **Paper, Theorem 1:** FlashAttention computes `O = softmax(QKᵀ)V` with `O(N²d)` FLOPs and requires only **`O(N)` additional memory** beyond inputs and output.


## 2. The Heart — Online Softmax

To avoid storing the `(T, T)` scores, we must compute the softmax **without ever seeing a whole row at once** — processing the row in *blocks* and carrying a tiny running summary. This is the **online softmax** (Milakov & Gimelshein, 2018), and it is the trick that makes everything else possible.

### The numerical-stability detour we can't skip

Softmax subtracts the row max for stability: `softmax(x)ᵢ = exp(xᵢ − m) / Σⱼ exp(xⱼ − m)` where `m = max(x)`. The problem: you need `m` (a property of the *whole* row) before you can start. Online softmax fixes this by keeping a **running max** `m` and a **running denominator** `ℓ`, and *correcting* them as each new block arrives.

### The recurrence

Suppose we have processed some blocks and hold `(m, ℓ)`. A new block of scores arrives with local max `m̃`. The paper's algebraic-aggregation identity (§3.1) updates the state:

$$ m^{\text{new}} = \max(m,\ \tilde m) $$

$$ \ell^{\text{new}} = e^{m - m^{\text{new}}}\,\ell \;+\; \sum_{\text{new block}} e^{s - m^{\text{new}}} $$

The factor `e^{m − m^new}` **rescales the old denominator** to the new reference max. That same correction factor must also rescale the partial output accumulator `O` — which is the only extra wrinkle when we fold `V` in. Let's verify the math numerically before trusting it.


![Online softmax running state](course/figures/fig_16b_online_softmax.png)

In [ ]:
import numpy as np
np.random.seed(0)

def naive_softmax(x: np.ndarray) -> np.ndarray:
    m = x.max(axis=-1, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=-1, keepdims=True)

def online_softmax_den(x: np.ndarray, block: int):
    '''Compute per-row (running max m, denominator l) block-by-block, never seeing a full row.'''
    n_rows, n_cols = x.shape
    m = np.full((n_rows, 1), -np.inf)
    l = np.zeros((n_rows, 1))
    for j in range(0, n_cols, block):
        xb = x[:, j:j + block]
        m_tilde = xb.max(axis=1, keepdims=True)
        m_new = np.maximum(m, m_tilde)
        l = np.exp(m - m_new) * l + np.exp(xb - m_new).sum(axis=1, keepdims=True)
        m = m_new
    return m, l

x = np.random.randn(4, 512) * 3.0          # 4 rows, 512 "keys"
m, l = online_softmax_den(x, block=64)     # processed in 8 blocks of 64
online_probs = np.exp(x - m) / l           # reconstruct full softmax from running stats
err = np.abs(online_probs - naive_softmax(x)).max()
print(f"online vs naive softmax max error: {err:.2e}  ->  {'PASS' if err < 1e-12 else 'FAIL'}")


The error is at floating-point noise level — **online softmax is exact**, not an approximation. Now fold in `V`: instead of just a denominator, accumulate the (unnormalized) output and apply the *same* rescaling correction. This is Flash Attention's forward pass, written in numpy so the algorithm is naked before we go to CUDA.


In [ ]:
def flash_forward_numpy(Q, K, V, Bc: int):
    '''Exact softmax(QK^T*scale)V computed block-by-block, never materializing the (N,N) scores.'''
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((N, d))
    m = np.full((N, 1), -np.inf)
    l = np.zeros((N, 1))
    for j in range(0, N, Bc):                      # loop over K/V column blocks
        Kj, Vj = K[j:j + Bc], V[j:j + Bc]
        S = (Q @ Kj.T) * scale                     # (N, Bc) score tile -- lives only here
        m_tilde = S.max(axis=1, keepdims=True)
        m_new = np.maximum(m, m_tilde)
        P = np.exp(S - m_new)                       # (N, Bc)
        corr = np.exp(m - m_new)                    # rescale factor for old accumulators
        l = corr * l + P.sum(axis=1, keepdims=True)
        O = corr * O + P @ Vj                       # rescale old O, add this block's contribution
        m = m_new
    return O / l                                   # deferred normalization, once, at the end

N, d = 128, 64
Q, K, V = (np.random.randn(N, d) for _ in range(3))
scale = 1.0 / np.sqrt(d)
O_ref = naive_softmax((Q @ K.T) * scale) @ V       # the textbook computation
O_flash = flash_forward_numpy(Q, K, V, Bc=16)
err = np.abs(O_flash - O_ref).max()
print(f"flash (numpy) vs naive attention max error: {err:.2e}  ->  {'PASS' if err < 1e-10 else 'FAIL'}")


### 2c. Tiling and Rescaling, Step by Step (the CS224n L18 view)

The `flash_forward_numpy` loop above *is* the whole algorithm — but the rescaling line (`O = corr*O + P@Vj`) is easy to accept without really seeing **why** it must be there. Stanford's CS224n Lecture 18 builds it up the most concrete way: start from a block split that **almost works**, watch it break, then patch it. We'll do the same with two key/value blocks.

**The whole-row computation** (one query row attending to all keys):

$$ S = QK^\top\cdot\text{scale}, \qquad A = \exp(S), \qquad \ell = \textstyle\sum_j A_j, \qquad O = \tfrac{A}{\ell}\,V $$

**1st attempt — split `K, V` into blocks and compute each block on its own.** With two blocks:

$$ A^{(1)} = \exp\!\big(Q (K^{(1)})^\top\big), \qquad A^{(2)} = \exp\!\big(Q (K^{(2)})^\top\big) $$

We want $O = \tfrac{A^{(1)}}{\ell}V^{(1)} + \tfrac{A^{(2)}}{\ell}V^{(2)}$ with $\ell = \sum A^{(1)} + \sum A^{(2)}$. **The catch:** the denominator $\ell$ depends on **both** blocks, so we cannot finish normalizing block 1 until block 2 has also been seen. That forces us to keep every block's scores around — exactly the `N×N` materialization we are trying to avoid.


**2nd attempt — rescaling.** Keep a *running* normalizer and a *running* output, and **fix up the output** every time the normalizer grows.

After block 1 (the denominator is only partial, so the output is "right shape, wrong scale"):

$$ \ell^{(1)} = \textstyle\sum A^{(1)}, \qquad O^{(1)} = \tfrac{A^{(1)}}{\ell^{(1)}}\,V^{(1)} $$

After block 2 (extend the denominator, then **rescale** the old output onto it before adding the new block):

$$ \ell^{(2)} = \ell^{(1)} + \textstyle\sum A^{(2)}, \qquad O^{(2)} = \frac{\ell^{(1)}}{\ell^{(2)}}\,O^{(1)} \;+\; \frac{A^{(2)}}{\ell^{(2)}}\,V^{(2)} $$

The factor $\ell^{(1)}/\ell^{(2)}$ **rescales the already-written output** from "divided by the partial sum $\ell^{(1)}$" to "divided by the full sum $\ell^{(2)}$." Now $O^{(2)}$ is the exact answer, and every step touched only one block — nothing `N×N` was ever stored. That is *tiling* (compute per block) + *rescaling* (correct the running output) — the two words in this section's title.

```mermaid
flowchart LR
  subgraph S1["Step 1 - block (K1,V1)"]
    a1["A1 = exp(Q K1^T)"] --> l1["l1 = sum A1"]
    l1 --> o1["O1 = (A1/l1) V1<br/>(wrong denominator)"]
  end
  subgraph S2["Step 2 - block (K2,V2)"]
    a2["A2 = exp(Q K2^T)"] --> l2["l2 = l1 + sum A2"]
  end
  o1 -- "rescale x l1/l2" --> o2["O2 = (l1/l2) O1 + (A2/l2) V2<br/>(exact answer)"]
  l2 --> o2
```

> **Bridge to the code above.** This lecture view drops the running **max** `m` to keep the algebra clean. `flash_forward_numpy` adds it so `exp()` can never overflow: the correction becomes `corr = e^{m_old − m_new}` (rescaling for a *new max*) layered on top of this same $\ell$-ratio idea. The `corr * O` and `corr * l` lines **are** the rescaling step you just derived, generalized for numerical stability.


In [ ]:
def attention_two_block_rescale(Q, K, V, split):
    '''The CS224n L18 two-attempt narrative: stream K/V in 2 blocks, rescaling the
    running output to the correct denominator. No running max yet (added later for stability).'''
    scale = 1.0 / np.sqrt(Q.shape[1])
    K1, K2, V1, V2 = K[:split], K[split:], V[:split], V[split:]

    # --- Step 1: block 1 locally (denominator is only partial -> output has the wrong scale) ---
    A1 = np.exp((Q @ K1.T) * scale)                 # (N, split)
    l1 = A1.sum(axis=1, keepdims=True)               # running normalizer after block 1
    O1 = (A1 / l1) @ V1                               # O^(1): right shape, wrong denominator

    # --- Step 2: extend the denominator, rescale O1 onto it, add block 2's contribution ---
    A2 = np.exp((Q @ K2.T) * scale)
    l2 = l1 + A2.sum(axis=1, keepdims=True)           # running normalizer after block 2
    O2 = (l1 / l2) * O1 + (A2 / l2) @ V2             # O^(2): rescale, then accumulate -> exact
    return O2

N, d = 64, 32
Q, K, V = (np.random.randn(N, d) for _ in range(3))
O_ref = naive_softmax((Q @ K.T) / np.sqrt(d)) @ V    # textbook attention over all keys at once
O_two = attention_two_block_rescale(Q, K, V, split=N // 2)
err = np.abs(O_two - O_ref).max()
print(f"two-block rescaled vs full attention max diff: {err:.2e}  ->  {'PASS' if err < 1e-10 else 'FAIL'}")


The two-block result matches full attention exactly. Notice what the rescaling bought us: we wrote `O^(1)` using only block 1, then **corrected it in place** when block 2 arrived — we never had to hold both blocks of scores at once. Generalize "two blocks" to "a loop over `Tc` blocks" and add the running max, and you are back at `flash_forward_numpy`.


### 2d. One Merge, by Hand, with Real Numbers

§2c dropped the running **max** to keep the algebra clean. But the max is exactly where learners get lost: *which* max, *what* gets rescaled, and *why `ℓ` and `O` are still "wrong" until the very last block.* So let's do **one** merge with concrete numbers and watch every running quantity update. (This is the worked example from the *"tiling algebra inside Flash Attention"* writeup, in our deferred-normalization convention.)

Take a single query row. We've already processed **block 1** and hold this running state:

$$ m = 5, \qquad \ell = 10, \qquad O_{\text{acc}} = [2,\ 3] $$

Here `O_acc` is the **unnormalized** accumulator (`Σ e^{s−m} · v`), *not* the answer — we have not divided by `ℓ` yet. Now **block 2** arrives. Suppose its scores give a local max `m̃ = 7`, and (already exponentiated at that local max) a block sum `ℓ̃ = 8` and a weighted-value contribution `P̃V = [1, 2]`.

**Step 1 — new reference max.** $m^{\text{new}} = \max(5, 7) = 7$. The reference rose by 2, so everything we stored relative to the old max `5` is now slightly too large.

**Step 2 — two correction factors.** Rescale the *old* state down to the new reference, and the *new* block (stored at its own local max `7`) onto it too:

$$ \alpha = e^{m - m^{\text{new}}} = e^{5-7} = e^{-2} \approx 0.135 \qquad\text{(shrinks the old running state)} $$
$$ \beta  = e^{\tilde m - m^{\text{new}}} = e^{7-7} = 1.0 \qquad\text{(new block needs no shrink — it set the new max)} $$

Both factors are in `(0, 1]` because `m^new ≥` either max. **`α` is the "rescale" of §2c, now generalized for a moving max.**

**Step 3 — update the running denominator and accumulator.**

$$ \ell^{\text{new}} = \alpha\,\ell + \beta\,\tilde\ell = 0.135(10) + 1.0(8) = 1.35 + 8 = 9.35 $$
$$ O_{\text{acc}}^{\text{new}} = \alpha\,O_{\text{acc}} + \beta\,\tilde P V = 0.135[2,3] + 1.0[1,2] = [1.27,\ 2.41] $$

**Step 4 — still don't divide yet.** `ℓ = 9.35` is the sum *so far*; a block 3 would grow it again. Only after the **last** block do we normalize once: `O = O_acc / ℓ`. That is why the chapter keeps `O_acc` unnormalized the whole way through — dividing early would just have to be undone.

> **Map to the code.** `flash_forward_numpy` folds `β` into `P` by computing `P = exp(S − m_new)` directly (so the new block is *already* at the new reference, `β=1` baked in), and writes `α` as `corr = exp(m − m_new)`. The lines `l = corr*l + P.sum()` and `O = corr*O + P@Vj` **are Steps 2-3 above.** The single final `return O / l` is Step 4.


In [ ]:
# Reproduce the by-hand merge above (blog's rounded numbers), step by step.
m, l, O_acc = 5.0, 10.0, np.array([2.0, 3.0])     # running state after block 1 (unnormalized)
m_tilde, l_tilde, PV = 7.0, 8.0, np.array([1.0, 2.0])  # block 2's local-max summaries

m_new = max(m, m_tilde)                            # Step 1: new reference max
alpha = np.exp(m - m_new)                          # Step 2: shrink old state
beta  = np.exp(m_tilde - m_new)                    #         place new block on new reference
l_new = alpha * l + beta * l_tilde                 # Step 3: running denominator
O_new = alpha * O_acc + beta * PV                  #         running (still unnormalized) accumulator
print(f"m_new={m_new}, alpha={alpha:.3f}, beta={beta:.3f}")
print(f"l_new={l_new:.4f}   O_acc_new={O_new}   ->  normalized answer O = {O_new / l_new}")
print(f"normalize-now would give {O_acc/l} for block 1 alone -- WRONG, that's why we defer\n")

# Now PROVE the merge is exact: build two raw blocks, summarize each, merge with the running
# max, and compare to a plain full-row softmax over both blocks concatenated.
np.random.seed(11)
d2 = 2
q  = np.random.randn(1, d2)
K1, V1 = np.random.randn(3, d2), np.random.randn(3, d2)
K2, V2 = np.random.randn(3, d2), np.random.randn(3, d2)
sc = 1.0 / np.sqrt(d2)
def summarize(Kb, Vb):                              # one block's (local max, sum, unnormalized P@V)
    S = (q @ Kb.T) * sc; mb = S.max(); P = np.exp(S - mb)
    return mb, P.sum(), P @ Vb
m1, l1, O1 = summarize(K1, V1)
m2, l2, O2 = summarize(K2, V2)
mn = max(m1, m2); a, b = np.exp(m1 - mn), np.exp(m2 - mn)
O_merged = (a * O1 + b * O2) / (a * l1 + b * l2)    # merge, then normalize once
O_ref = naive_softmax((q @ np.vstack([K1, K2]).T) * sc) @ np.vstack([V1, V2])
err = np.abs(O_merged - O_ref).max()
print(f"merged-with-max vs full softmax over both blocks: {err:.2e}  ->  {'PASS' if err < 1e-12 else 'FAIL'}")


Notice the payoff of the deferred-normalization view: when block 2 pushed the max from 5 to 7, the *old* contribution was automatically **down-weighted** by `α ≈ 0.135` — the running output stays an exact weighted average over *all tokens seen so far*, with no `N×N` matrix ever held. Next we map this exact loop onto GPU tiles.


## 3. Tiling — Mapping Online Softmax onto the GPU

The numpy version proves the algorithm; now we make it GPU-shaped. The plan: split Q into **row blocks** of `Br` and K, V into **column blocks** of `Bc`. Each Q row-block streams over all K/V column-blocks, holding its `(O, m, ℓ)` running state in **SRAM** the whole time. The score tile `Sᵢⱼ` is computed, consumed, and thrown away — it never reaches HBM.

![Flash Attention tiling dataflow](course/figures/fig_16b_tiling_grid.png)

**Watch it run.** The animation below streams the K/V blocks in one at a time (the cs224n L18 / Francisco Massa animation, redrawn with real numbers). Only the red score block exists in SRAM at any moment; after each block the running denominator `ℓ` grows and the partial output `O/ℓ` is **rescaled** — converging, block by block, onto the exact `softmax(QKᵀ)V` (grey ticks). This is §2c's two-attempt story, now in motion.

![Tiling + rescaling, animated](course/figures/fig_16b_tiling_anim.gif)


### How the block sizes are chosen

The blocks must fit in SRAM of size `M`. The paper sets:

$$ B_c = \left\lceil \frac{M}{4d} \right\rceil, \qquad B_r = \min\!\left(\left\lceil \frac{M}{4d}\right\rceil,\ d\right) $$

The `4d` accounts for the four tiles that must coexist in SRAM (Q, K, V, and the score block), and capping `Br` at `d` keeps the score tile from dominating.

### Algorithm 1 (FlashAttention forward), structure

> 1. Set `Bc = ⌈M/4d⌉`, `Br = min(⌈M/4d⌉, d)`.
> 2. Initialize `O = 0`, `ℓ = 0`, `m = −∞` in HBM.
> 3. Split Q into `Tr = ⌈N/Br⌉` row blocks; K, V into `Tc = ⌈N/Bc⌉` column blocks.
> 5. **for** `1 ≤ j ≤ Tc`: load `Kⱼ, Vⱼ` to SRAM.
> 7.   **for** `1 ≤ i ≤ Tr`: load `Qᵢ, Oᵢ, ℓᵢ, mᵢ` to SRAM.
> 9.     `Sᵢⱼ = Qᵢ Kⱼᵀ`.
> 10.    `m̃ᵢⱼ = rowmax(Sᵢⱼ)`, `P̃ᵢⱼ = exp(Sᵢⱼ − m̃ᵢⱼ)`, `ℓ̃ᵢⱼ = rowsum(P̃ᵢⱼ)`.
> 11.    `mᵢⁿᵉʷ = max(mᵢ, m̃ᵢⱼ)`, `ℓᵢⁿᵉʷ = e^{mᵢ−mᵢⁿᵉʷ}ℓᵢ + e^{m̃ᵢⱼ−mᵢⁿᵉʷ}ℓ̃ᵢⱼ`.
> 12.    `Oᵢ ← diag(ℓᵢⁿᵉʷ)⁻¹ (diag(ℓᵢ) e^{mᵢ−mᵢⁿᵉʷ} Oᵢ + e^{m̃ᵢⱼ−mᵢⁿᵉʷ} P̃ᵢⱼ Vⱼ)`.
> 13.    Write `ℓᵢ ← ℓᵢⁿᵉʷ`, `mᵢ ← mᵢⁿᵉʷ`.

Our CUDA kernel below uses the slightly cleaner **deferred-normalization** form (divide by `ℓ` once at the very end, as in our numpy version and in FlashAttention-2) — mathematically identical, fewer divisions.


## 4. Walkthrough — How `flash-attention-minimal` Lays It Out

Before we write our own, here is the launch structure of [`flash-attention-minimal`](https://github.com/tspeterkim/flash-attention-minimal) (the cleanest reference), paraphrased:

```cpp
dim3 grid_dim(B, nh);   // one thread block per (batch, head)
dim3 block_dim(Bc);     // Bc threads per block -- one thread per query row in a tile

// shared memory holds Q, K, V tiles + the score tile:
//   sram = 3 * Bc * d * sizeof(float)  +  Bc * Br * sizeof(float)
float* Qi = sram;
float* Kj = &sram[tile_size];
float* Vj = &sram[tile_size * 2];
float* S  = &sram[tile_size * 3];

for (int j = 0; j < Tc; j++) {        // outer: K/V column blocks
    // load Kj, Vj into SRAM
    for (int i = 0; i < Tr; i++) {     // inner: Q row blocks
        // S = scale * Qi . Kj^T ; row_m = rowmax(S)
        float row_m_new = max(row_m_prev, row_m);
        float row_l_new = __expf(row_m_prev - row_m_new) * row_l_prev
                        + __expf(row_m       - row_m_new) * row_l;
        // O = (1/row_l_new) * ( row_l_prev * e^{m_prev-m_new} * O  +  e^{m-m_new} * P.V )
    }
}
```

Notice the loop order: **K/V outer, Q inner** — this is FlashAttention-**1**. (FlashAttention-2 swaps them; §7.) Our kernel below uses the FA-2-friendly **Q-outer** mapping — one thread block per Q row-block — because it makes the per-row running state purely thread-local and easier to read.


### 4a. Our forward kernel — one thread per query row

We specialize to a small, self-contained case so the whole thing compiles and verifies in one file: `N=64`, `d=64`, block sizes `Br=Bc=16`. Each **thread block** owns one Q row-block (`Br` rows); each **thread** owns one query row and keeps its `(o[d], m, ℓ)` state in registers. K/V column blocks are staged through shared memory.


In [ ]:
%%writefile course/ch16b_build/flash_fwd.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

#define N  64     // sequence length
#define D  64     // head dimension
#define BC 16     // K/V column-block size
#define BR 16     // Q row-block size

// One thread block per Q row-block (grid = Tr); one thread per query row (blockDim = BR).
__global__ void flash_fwd(const float* Q, const float* K, const float* V, float* O, float scale) {
    int row = blockIdx.x * BR + threadIdx.x;          // global query row this thread owns

    float q[D], o[D];                                  // this row's query + output accumulator
    for (int k = 0; k < D; k++) { q[k] = Q[row * D + k]; o[k] = 0.0f; }
    float m = -1e30f, l = 0.0f;                         // running max and denominator

    __shared__ float Ks[BC * D];                       // staged K/V column block
    __shared__ float Vs[BC * D];

    int Tc = N / BC;
    for (int j = 0; j < Tc; j++) {
        // cooperatively load Kj, Vj (BC*D elements) into SRAM
        for (int idx = threadIdx.x; idx < BC * D; idx += BR) {
            Ks[idx] = K[j * BC * D + idx];
            Vs[idx] = V[j * BC * D + idx];
        }
        __syncthreads();

        // 1) score tile for this row: s[c] = scale * (q . Kj[c]); track block max
        float s[BC], m_tilde = -1e30f;
        for (int c = 0; c < BC; c++) {
            float dot = 0.0f;
            for (int k = 0; k < D; k++) dot += q[k] * Ks[c * D + k];
            dot *= scale;
            s[c] = dot;
            if (dot > m_tilde) m_tilde = dot;
        }
        // 2) online-softmax update of (m, l) and rescale O
        float m_new = fmaxf(m, m_tilde);
        float corr  = __expf(m - m_new);
        float l_tilde = 0.0f;
        for (int c = 0; c < BC; c++) { s[c] = __expf(s[c] - m_new); l_tilde += s[c]; }
        l = corr * l + l_tilde;
        for (int k = 0; k < D; k++) {
            float pv = 0.0f;
            for (int c = 0; c < BC; c++) pv += s[c] * Vs[c * D + k];
            o[k] = corr * o[k] + pv;
        }
        m = m_new;
        __syncthreads();                                // before overwriting Ks/Vs next iter
    }
    for (int k = 0; k < D; k++) O[row * D + k] = o[k] / l;   // deferred normalization
}

int main(void) {
    const int sz = N * D;
    float *hQ = (float*)malloc(sz*4), *hK = (float*)malloc(sz*4), *hV = (float*)malloc(sz*4);
    float *hO = (float*)malloc(sz*4), *hRef = (float*)malloc(sz*4);
    for (int i = 0; i < sz; i++) {                       // deterministic pseudo-random inputs
        hQ[i] = sinf(0.1f*i);  hK[i] = cosf(0.07f*i);  hV[i] = sinf(0.03f*i + 1.0f);
    }
    float scale = 1.0f / sqrtf((float)D);

    // CPU reference: full softmax(QK^T*scale) @ V
    for (int i = 0; i < N; i++) {
        float s[N], mx = -1e30f, sum = 0.0f;
        for (int j = 0; j < N; j++) {
            float dot = 0.0f; for (int k = 0; k < D; k++) dot += hQ[i*D+k]*hK[j*D+k];
            s[j] = dot*scale; if (s[j] > mx) mx = s[j];
        }
        for (int j = 0; j < N; j++) { s[j] = expf(s[j]-mx); sum += s[j]; }
        for (int k = 0; k < D; k++) {
            float acc = 0.0f; for (int j = 0; j < N; j++) acc += s[j]*hV[j*D+k];
            hRef[i*D+k] = acc / sum;
        }
    }

    float *dQ,*dK,*dV,*dO;
    cudaMalloc(&dQ,sz*4); cudaMalloc(&dK,sz*4); cudaMalloc(&dV,sz*4); cudaMalloc(&dO,sz*4);
    cudaMemcpy(dQ,hQ,sz*4,cudaMemcpyHostToDevice);
    cudaMemcpy(dK,hK,sz*4,cudaMemcpyHostToDevice);
    cudaMemcpy(dV,hV,sz*4,cudaMemcpyHostToDevice);
    flash_fwd<<<N/BR, BR>>>(dQ,dK,dV,dO,scale);
    cudaMemcpy(hO,dO,sz*4,cudaMemcpyDeviceToHost);

    float maxerr = 0.0f;
    for (int i = 0; i < sz; i++) { float e = fabsf(hO[i]-hRef[i]); if (e > maxerr) maxerr = e; }
    printf("flash forward (CUDA) vs CPU reference max diff: %.2e  %s\n",
           maxerr, maxerr < 1e-4 ? "PASS" : "FAIL");
    cudaFree(dQ);cudaFree(dK);cudaFree(dV);cudaFree(dO);
    free(hQ);free(hK);free(hV);free(hO);free(hRef);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch16b_build/flash_fwd course/ch16b_build/flash_fwd.cu && ./course/ch16b_build/flash_fwd


**That is a real Flash Attention forward kernel.** It computes exact attention, the `(N,N)` scores never exist in HBM (only a `BC`-wide tile in registers/SRAM at a time), and the only persistent state per row is `(m, ℓ, o[D])`. The production kernels add: multiple heads/batches in the grid, vectorized loads, warp-level matmuls (tensor cores), and tuned block sizes — but the algorithm on the page above is the same one cuDNN runs.


## 5. Why It's Faster — IO Complexity

The payoff is provable. Counting HBM accesses (the paper's Theorem 2):

| | HBM accesses |
|---|---|
| Standard attention | `Θ(Nd + N²)` |
| **FlashAttention** | `Θ(N²d² M⁻¹)` |

Since SRAM size `M` (~100 KB) is many times larger than `d²` (for `d` = 64-128), `N²d²/M ≪ N²`, so Flash Attention makes **many times fewer** HBM round trips — even though it does *more* arithmetic (recomputation). On GPT-2 medium the paper measures:

![FlashAttention benchmark, paper Fig. 2](course/figures/fig_16b_benchmark.png)

HBM traffic drops `40.3 → 4.4 GB` (~9x) and forward+backward runtime `41.7 → 7.3 ms` (~5.7x) — *despite* FlashAttention's higher FLOP count (75.2 vs 66.6 GFLOPs). The lesson, one more time: on modern GPUs, **moving memory is the cost; arithmetic is nearly free.**

> **Proposition 3 (lower bound):** no exact-attention algorithm can beat `Θ(N²d²M⁻¹)` HBM accesses for all `M` — FlashAttention is asymptotically optimal in memory traffic.


## 6. The Backward Pass — Recompute, Don't Store

Training needs gradients `dQ, dK, dV` from `dO`. The naive backward needs `S` and `P` (each `N×N`) — exactly the tensors Flash Attention refused to store. Two ideas rescue it:

1. **Recomputation (selective checkpointing).** We *recompute* `S` and `P` block-by-block in SRAM from the saved `Q, K, V` and the saved softmax stats `(m, ℓ)` — same tiling as the forward. More FLOPs, but no `N²` HBM reads, which (again) wins.
2. **The `Dᵢ` trick.** The softmax Jacobian makes `dS` look expensive, but it collapses to one scalar per row.

### Recomputation, step by step (the CS224n L18 view)

The lecture frames the backward as three moves — the mirror of the forward's *load → compute → rescale*:

1. **Save only the `O(N)` stats in the forward.** Keep the per-row softmax normalizer `ℓ` (and max `m`) — `size N`, not `N×N`. Throw the scores `S` and probabilities `P` away. *(Production FlashAttention is even tighter: it fuses the two into a single per-row scalar `Lᵢ = mᵢ + log(ℓᵢ)`, the **log-sum-exp**, so one `O(N)` vector — not two — is all the forward hands to the backward.)*
2. **Recompute the tile in SRAM.** In the backward, reload `Q, K, V` blocks and rebuild `Sᵢⱼ` and `Pᵢⱼ = exp(Sᵢⱼ − m)/ℓ` on chip from those saved stats — same tiling as the forward, no `N×N` HBM reads. (With the fused stat this is the single clean line `Pᵢⱼ = exp(Sᵢⱼ − Lᵢ)`, since `Lᵢ` already carries both the max and the log-denominator.)
3. **Form the gradients with `D`.** Use `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)` to get `dS` per tile, then accumulate `dQ, dK, dV`.

This *adds* FLOPs (we compute `S, P` twice) yet still wins: the forward+backward HBM traffic drops `40.3 → 4.4 GB` and runtime `41.7 → 7.3 ms` on GPT-2 medium (the §5 benchmark) **even though** FlashAttention does more arithmetic — the same "memory is the cost, math is free" trade as the forward.

![Backward: recomputation + the D trick](course/figures/fig_16b_backward.png)


### The gradient identities (paper, Appendix B)

With `O = P V` and `P = softmax(S)` row-wise, reverse-mode gives:

$$ dV = P^\top\, dO \qquad dP = dO\, V^\top $$

The softmax backward for a row is `dS = P ∘ (dP − rowsum(P ∘ dP))`. The clever part: define

$$ D_i = \text{rowsum}(dO_i \circ O_i) $$

Then `rowsum(P ∘ dP) = D` exactly (proof below), so:

$$ dS = P \circ (dP - D), \qquad dQ = (dS\,K)\cdot\text{scale}, \qquad dK = (dS^\top Q)\cdot\text{scale} $$

`D` is a single number per query row — cheap to compute and broadcast, no `N×N` intermediate needed.


### 6a. Why `rowsum(P ∘ dP) = rowsum(dO ∘ O)`

For row `i`: since `Oᵢ = Σⱼ Pᵢⱼ Vⱼ` and `dPᵢⱼ = dOᵢ · Vⱼ`,

$$ \sum_j P_{ij}\, dP_{ij} = \sum_j P_{ij}\,(dO_i \cdot V_j) = dO_i \cdot \Big(\sum_j P_{ij} V_j\Big) = dO_i \cdot O_i = \text{rowsum}(dO_i \circ O_i). $$

So the expensive-looking `rowsum(P ∘ dP)` equals `Dᵢ`, computable straight from the **outputs and their gradients** — no need to keep `P` or `dP` around. Let's verify the whole backward numerically: an FA-style blocked backward (recompute `P`, use `D`) against both a naive materialized backward and a finite-difference check.


In [ ]:
def backward_naive(Q, K, V, dO):
    '''Reference backward: materialize P, use the full softmax Jacobian.'''
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    S = (Q @ K.T) * scale
    P = naive_softmax(S)
    O = P @ V
    dV = P.T @ dO
    dP = dO @ V.T
    dS = P * (dP - (P * dP).sum(axis=1, keepdims=True))   # softmax Jacobian, row-wise
    dQ = (dS @ K) * scale
    dK = (dS.T @ Q) * scale
    return dQ, dK, dV, O

def backward_flash(Q, K, V, O, dO, Bc):
    '''FA-style: recompute S/P per block, use D = rowsum(dO o O); never store the (N,N) matrices.'''
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    D_row = (dO * O).sum(axis=1, keepdims=True)            # the D trick: one scalar per row
    dQ = np.zeros_like(Q); dK = np.zeros_like(K); dV = np.zeros_like(V)
    # the softmax denominators/maxes are the O(N) stats saved by the forward pass
    full = (Q @ K.T) * scale
    m = full.max(axis=1, keepdims=True)
    _, l = online_softmax_den(full, block=Bc)
    for j in range(0, N, Bc):                              # block over keys
        Kj, Vj = K[j:j + Bc], V[j:j + Bc]
        S = (Q @ Kj.T) * scale                            # recomputed score tile (N, Bc)
        P = np.exp(S - m) / l                              # exact softmax probs for this tile
        dV[j:j + Bc] = P.T @ dO
        dP = dO @ Vj.T                                     # (N, Bc)
        dS = P * (dP - D_row)                              # uses D, no rowsum over the full row
        dQ += (dS @ Kj) * scale
        dK[j:j + Bc] = (dS.T @ Q) * scale
    return dQ, dK, dV

np.random.seed(1)
N, d = 96, 32
Q, K, V = (np.random.randn(N, d) for _ in range(3))
dO = np.random.randn(N, d)
dQn, dKn, dVn, O = backward_naive(Q, K, V, dO)
dQf, dKf, dVf = backward_flash(Q, K, V, O, dO, Bc=16)
for name, a, b in [("dQ", dQf, dQn), ("dK", dKf, dKn), ("dV", dVf, dVn)]:
    e = np.abs(a - b).max()
    print(f"{name}: flash vs naive max diff {e:.2e}  ->  {'PASS' if e < 1e-9 else 'FAIL'}")


In [ ]:
# Independent check: finite differences on a scalar loss = sum(W * O), so dO = W.
def attention_out(Q, K, V):
    scale = 1.0 / np.sqrt(Q.shape[1])
    return naive_softmax((Q @ K.T) * scale) @ V

W = np.random.randn(N, d)
dO_fd = W.copy()
_, _, _, O0 = backward_naive(Q, K, V, dO_fd)
dQ_an, _, _ = backward_flash(Q, K, V, O0, dO_fd, Bc=16)

eps = 1e-5
dQ_fd = np.zeros_like(Q)
for i in range(N):
    for k in range(d):
        Qp = Q.copy(); Qp[i, k] += eps
        Qm = Q.copy(); Qm[i, k] -= eps
        dQ_fd[i, k] = ((W * attention_out(Qp, K, V)).sum() - (W * attention_out(Qm, K, V)).sum()) / (2 * eps)
e = np.abs(dQ_an - dQ_fd).max()
print(f"dQ: analytic (flash) vs finite-difference max diff {e:.2e}  ->  {'PASS' if e < 1e-5 else 'FAIL'}")


Both checks pass: the FA backward — recomputing `P` block-by-block and using `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)` — produces the exact gradients, with only `O(N)` saved state (`m, ℓ`) instead of the `N×N` matrices. Porting this to CUDA follows the same shape as the forward kernel (a block loop with running accumulators); [`66RING/tiny-flash-attention`](https://github.com/66RING/tiny-flash-attention) and [`sonnyli/flash_attention_from_scratch`](https://github.com/sonnyli/flash_attention_from_scratch) have complete CUDA backward kernels worth reading next.


## 7. FlashAttention-1 vs FlashAttention-2

[FlashAttention-2](https://arxiv.org/abs/2307.08691) keeps the algorithm but reorganizes it for the hardware, hitting ~2x the speed of FA-1 (50-73% of A100 peak FLOPs). Three changes:

| Change | FlashAttention-1 | FlashAttention-2 |
|---|---|---|
| **Non-matmul FLOPs** | rescales `O` by `1/ℓ` *every* block | defers the `1/ℓ` division to the **end** (what our kernel already does) — fewer non-matmul ops, which run far slower than tensor-core matmuls |
| **Loop order / parallelism** | outer loop over **K/V**; parallel over batch x heads | outer loop over **Q**; **also parallelizes over query blocks (sequence length)** -> more thread blocks, higher occupancy for long sequences |
| **Warp work-partitioning** | splits **K** across warps ("split-K") -> warps must share partial results through shared memory | splits **Q** across warps ("split-Q") -> each warp owns whole query rows, **no inter-warp shared-memory communication** |

**Why "non-matmul FLOPs" is its own category — and not a memory issue.** §5 taught one bottleneck (HBM traffic); the first row of this table is a *different* one, on the **compute** side. A GPU has two kinds of arithmetic units with wildly different throughput:

- **Tensor cores** — dedicated matrix-multiply hardware, *enormously* high throughput. The `QKᵀ` and `PV` matmuls run here.
- **CUDA cores / SFUs** — general-purpose units that do `exp`, division, comparisons, the rescaling `corr*O`. Often **an order of magnitude (or more) lower throughput** per op.

So raw **FLOP count misleads**: a matmul FLOP and a division FLOP cost very different amounts of *time*. If tensor cores chew through matmul FLOPs ~16x faster than CUDA cores do non-matmul FLOPs, then a division that is only ~5% of the FLOP *count* can eat a large slice of the *wall-clock* — and stall the fast tensor cores while they wait. That is why FA-2 obsesses over non-matmul ops specifically: deferring the `1/ℓ` division pulls those slow-per-op instructions out of the hot inner loop. It is the **same lesson as §5, one level in** — don't count operations, count *time on whichever resource is the bottleneck* (there it was bytes over HBM; here it is ops on the slow compute units).

Our kernel already adopts FA-2's deferred normalization and Q-outer mapping (one block per Q row-block). The remaining FA-2 wins — splitting Q across *warps* within a block and using tensor-core matmuls — are what separate a teaching kernel from a production one. (In fact our teaching kernel runs **both** dot products as scalar CUDA-core loops, leaving the tensor cores idle — moving those matmuls onto tensor cores is its single biggest missing optimization.)

> FlashAttention-**3** (2024) goes further, exploiting Hopper's async copies (TMA) and FP8; same core algorithm, newer silicon.


### Going to production

Do **not** ship the kernel above. It's a teaching kernel: fixed tiny sizes, scalar loads, no tensor cores, fp32 only. In real code you have three good options, in order of effort:

1. **`F.scaled_dot_product_attention(q, k, v, is_causal=True)`** (PyTorch) — dispatches to a fused FlashAttention/cuDNN backend automatically. Use this unless you have a reason not to.
2. **The `flash-attn` package** (Dao's official CUDA kernels, FA-2/FA-3) — for custom masks, sliding windows, etc.
3. **cuDNN's fused attention** — what `llm.c` calls in [`llmc/cudnn_att.h`](llmc/cudnn_att.h) (Chapter 16).

Write your own only to *learn*, to support a layout the libraries don't, or to chase the last 10% on a specific GPU.


## 8. Exercises

Four exercises, increasing in difficulty. Each has a runnable check and a solution below it. Try before peeking.


### Exercise 1 — Merge two softmax states

The core primitive of online softmax is *merging two already-summarized blocks*. Given `(m_a, l_a)` and `(m_b, l_b)` for two disjoint chunks of the same row, return the combined `(m, l)` — without seeing the raw scores again. Fill in the TODO.


In [ ]:
def merge_softmax_state(m_a, l_a, m_b, l_b):
    # TODO: combine two (max, denominator) summaries into one.
    # hint: pick the new max, then rescale BOTH denominators to it before adding.
    m_new = 0.0   # replace
    l_new = 0.0   # replace
    return m_new, l_new

# check: split a row in half, merge the halves, compare to processing it whole
np.random.seed(7)
row = np.random.randn(1, 256) * 2.5
ma, la = online_softmax_den(row[:, :128], block=32)
mb, lb = online_softmax_den(row[:, 128:], block=32)
m_merged, l_merged = merge_softmax_state(ma, la, mb, lb)
m_whole, l_whole = online_softmax_den(row, block=32)
ok = np.allclose(m_merged, m_whole) and np.allclose(l_merged, l_whole)
print("PASS" if ok else "FAIL (still the stub)")


#### Solution

In [ ]:
def merge_softmax_state(m_a, l_a, m_b, l_b):
    m_new = np.maximum(m_a, m_b)
    l_new = np.exp(m_a - m_new) * l_a + np.exp(m_b - m_new) * l_b
    return m_new, l_new

ma, la = online_softmax_den(row[:, :128], block=32)
mb, lb = online_softmax_den(row[:, 128:], block=32)
m_merged, l_merged = merge_softmax_state(ma, la, mb, lb)
m_whole, l_whole = online_softmax_den(row, block=32)
assert np.allclose(m_merged, m_whole) and np.allclose(l_merged, l_whole)
print("PASS")


### Exercise 2 — Pick the block sizes

You have `M = 49152` floats of usable SRAM (48 KB / 4 bytes) and head dim `d = 64`. Using the paper's formulas `Bc = ⌈M/4d⌉` and `Br = min(⌈M/4d⌉, d)`, compute the block sizes and confirm the four tiles (Q, K, V, S) fit in `M`. Fill in the TODO.


In [ ]:
import math
def choose_block_sizes(M, d):
    # TODO: compute Bc and Br from the paper's formulas.
    Bc = 0   # replace with ceil(M / (4*d))
    Br = 0   # replace with min(ceil(M / (4*d)), d)
    return Bc, Br

M, d = 49152, 64
Bc, Br = choose_block_sizes(M, d)
# four tiles must fit: Q(Br*d) + K(Bc*d) + V(Bc*d) + S(Br*Bc)
floats_needed = Br*d + Bc*d + Bc*d + Br*Bc
print(f"Bc={Bc}, Br={Br}, tiles use {floats_needed} floats (budget {M}) -> "
      f"{'PASS' if Bc>0 and floats_needed <= M else 'FAIL (still the stub)'}")


#### Solution

In [ ]:
def choose_block_sizes(M, d):
    Bc = math.ceil(M / (4 * d))
    Br = min(Bc, d)
    return Bc, Br

Bc, Br = choose_block_sizes(49152, 64)
floats_needed = Br*64 + Bc*64 + Bc*64 + Br*Bc
assert Bc == 192 and Br == 64 and floats_needed <= 49152
print(f"Bc={Bc}, Br={Br}, tiles use {floats_needed} floats <= 49152  PASS")


### Exercise 3 — Add a causal mask to the forward kernel

Autoregressive attention forbids a query at position `i` from attending to keys at `j > i`. In the tiled kernel, a key's **global** index is `j*BC + c`. Mask it by treating its score as `−∞` (so `exp(...) = 0`) whenever it lies in the future. Complete the kernel; the CPU reference is already causal.


In [ ]:
%%writefile course/ch16b_build/exercise_causal.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>
#define N 64
#define D 64
#define BC 16
#define BR 16

__global__ void flash_fwd_causal(const float* Q, const float* K, const float* V, float* O, float scale) {
    int row = blockIdx.x * BR + threadIdx.x;
    float q[D], o[D];
    for (int k=0;k<D;k++){ q[k]=Q[row*D+k]; o[k]=0.0f; }
    float m=-1e30f, l=0.0f;
    __shared__ float Ks[BC*D]; __shared__ float Vs[BC*D];
    int Tc=N/BC;
    for (int j=0;j<Tc;j++){
        for (int idx=threadIdx.x; idx<BC*D; idx+=BR){ Ks[idx]=K[j*BC*D+idx]; Vs[idx]=V[j*BC*D+idx]; }
        __syncthreads();
        float s[BC], m_tilde=-1e30f;
        for (int c=0;c<BC;c++){
            int key_idx = j*BC + c;
            float dot=0.0f; for(int k=0;k<D;k++) dot += q[k]*Ks[c*D+k];
            dot *= scale;
            // TODO: if key_idx > row, this key is in the future -> set dot = -1e30f (masked)
            s[c]=dot; if(dot>m_tilde) m_tilde=dot;
        }
        float m_new=fmaxf(m,m_tilde), corr=__expf(m-m_new), l_tilde=0.0f;
        for(int c=0;c<BC;c++){ s[c]=__expf(s[c]-m_new); l_tilde+=s[c]; }
        l=corr*l+l_tilde;
        for(int k=0;k<D;k++){ float pv=0.0f; for(int c=0;c<BC;c++) pv+=s[c]*Vs[c*D+k]; o[k]=corr*o[k]+pv; }
        m=m_new; __syncthreads();
    }
    for(int k=0;k<D;k++) O[row*D+k]=o[k]/l;
}

int main(void){
    const int sz=N*D;
    float *hQ=(float*)malloc(sz*4),*hK=(float*)malloc(sz*4),*hV=(float*)malloc(sz*4);
    float *hO=(float*)malloc(sz*4),*hRef=(float*)malloc(sz*4);
    for(int i=0;i<sz;i++){ hQ[i]=sinf(0.1f*i); hK[i]=cosf(0.07f*i); hV[i]=sinf(0.03f*i+1.0f); }
    float scale=1.0f/sqrtf((float)D);
    for(int i=0;i<N;i++){                                  // CAUSAL CPU reference
        float s[N],mx=-1e30f,sum=0.0f;
        for(int j=0;j<N;j++){
            if(j>i){ s[j]=-1e30f; continue; }
            float dot=0.0f; for(int k=0;k<D;k++) dot+=hQ[i*D+k]*hK[j*D+k];
            s[j]=dot*scale; if(s[j]>mx) mx=s[j];
        }
        for(int j=0;j<N;j++){ s[j]=(j>i)?0.0f:expf(s[j]-mx); sum+=s[j]; }
        for(int k=0;k<D;k++){ float acc=0.0f; for(int j=0;j<N;j++) acc+=s[j]*hV[j*D+k]; hRef[i*D+k]=acc/sum; }
    }
    float *dQ,*dK,*dV,*dO; cudaMalloc(&dQ,sz*4);cudaMalloc(&dK,sz*4);cudaMalloc(&dV,sz*4);cudaMalloc(&dO,sz*4);
    cudaMemcpy(dQ,hQ,sz*4,cudaMemcpyHostToDevice);cudaMemcpy(dK,hK,sz*4,cudaMemcpyHostToDevice);cudaMemcpy(dV,hV,sz*4,cudaMemcpyHostToDevice);
    flash_fwd_causal<<<N/BR,BR>>>(dQ,dK,dV,dO,scale);
    cudaMemcpy(hO,dO,sz*4,cudaMemcpyDeviceToHost);
    float maxerr=0.0f; for(int i=0;i<sz;i++){ float e=fabsf(hO[i]-hRef[i]); if(e>maxerr)maxerr=e; }
    printf("causal flash vs causal CPU ref max diff: %.2e  %s\n", maxerr, maxerr<1e-4?"PASS":"FAIL");
    cudaFree(dQ);cudaFree(dK);cudaFree(dV);cudaFree(dO); free(hQ);free(hK);free(hV);free(hO);free(hRef);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch16b_build/exercise_causal course/ch16b_build/exercise_causal.cu && ./course/ch16b_build/exercise_causal


#### Solution

In [ ]:
%%writefile course/ch16b_build/exercise_causal_sol.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>
#define N 64
#define D 64
#define BC 16
#define BR 16

__global__ void flash_fwd_causal(const float* Q, const float* K, const float* V, float* O, float scale) {
    int row = blockIdx.x * BR + threadIdx.x;
    float q[D], o[D];
    for (int k=0;k<D;k++){ q[k]=Q[row*D+k]; o[k]=0.0f; }
    float m=-1e30f, l=0.0f;
    __shared__ float Ks[BC*D]; __shared__ float Vs[BC*D];
    int Tc=N/BC;
    for (int j=0;j<Tc;j++){
        for (int idx=threadIdx.x; idx<BC*D; idx+=BR){ Ks[idx]=K[j*BC*D+idx]; Vs[idx]=V[j*BC*D+idx]; }
        __syncthreads();
        float s[BC], m_tilde=-1e30f;
        for (int c=0;c<BC;c++){
            int key_idx = j*BC + c;
            float dot=0.0f; for(int k=0;k<D;k++) dot += q[k]*Ks[c*D+k];
            dot *= scale;
            if (key_idx > row) dot = -1e30f;            // causal mask: future keys excluded
            s[c]=dot; if(dot>m_tilde) m_tilde=dot;
        }
        float m_new=fmaxf(m,m_tilde), corr=__expf(m-m_new), l_tilde=0.0f;
        for(int c=0;c<BC;c++){ s[c]=__expf(s[c]-m_new); l_tilde+=s[c]; }
        l=corr*l+l_tilde;
        for(int k=0;k<D;k++){ float pv=0.0f; for(int c=0;c<BC;c++) pv+=s[c]*Vs[c*D+k]; o[k]=corr*o[k]+pv; }
        m=m_new; __syncthreads();
    }
    for(int k=0;k<D;k++) O[row*D+k]=o[k]/l;
}

int main(void){
    const int sz=N*D;
    float *hQ=(float*)malloc(sz*4),*hK=(float*)malloc(sz*4),*hV=(float*)malloc(sz*4);
    float *hO=(float*)malloc(sz*4),*hRef=(float*)malloc(sz*4);
    for(int i=0;i<sz;i++){ hQ[i]=sinf(0.1f*i); hK[i]=cosf(0.07f*i); hV[i]=sinf(0.03f*i+1.0f); }
    float scale=1.0f/sqrtf((float)D);
    for(int i=0;i<N;i++){
        float s[N],mx=-1e30f,sum=0.0f;
        for(int j=0;j<N;j++){
            if(j>i){ s[j]=-1e30f; continue; }
            float dot=0.0f; for(int k=0;k<D;k++) dot+=hQ[i*D+k]*hK[j*D+k];
            s[j]=dot*scale; if(s[j]>mx) mx=s[j];
        }
        for(int j=0;j<N;j++){ s[j]=(j>i)?0.0f:expf(s[j]-mx); sum+=s[j]; }
        for(int k=0;k<D;k++){ float acc=0.0f; for(int j=0;j<N;j++) acc+=s[j]*hV[j*D+k]; hRef[i*D+k]=acc/sum; }
    }
    float *dQ,*dK,*dV,*dO; cudaMalloc(&dQ,sz*4);cudaMalloc(&dK,sz*4);cudaMalloc(&dV,sz*4);cudaMalloc(&dO,sz*4);
    cudaMemcpy(dQ,hQ,sz*4,cudaMemcpyHostToDevice);cudaMemcpy(dK,hK,sz*4,cudaMemcpyHostToDevice);cudaMemcpy(dV,hV,sz*4,cudaMemcpyHostToDevice);
    flash_fwd_causal<<<N/BR,BR>>>(dQ,dK,dV,dO,scale);
    cudaMemcpy(hO,dO,sz*4,cudaMemcpyDeviceToHost);
    float maxerr=0.0f; for(int i=0;i<sz;i++){ float e=fabsf(hO[i]-hRef[i]); if(e>maxerr)maxerr=e; }
    printf("causal flash vs causal CPU ref max diff: %.2e  %s\n", maxerr, maxerr<1e-4?"PASS":"FAIL");
    cudaFree(dQ);cudaFree(dK);cudaFree(dV);cudaFree(dO); free(hQ);free(hK);free(hV);free(hO);free(hRef);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch16b_build/exercise_causal_sol course/ch16b_build/exercise_causal_sol.cu && ./course/ch16b_build/exercise_causal_sol


### Exercise 4 — Compute `D` two ways and confirm they match

The backward's key shortcut is `Dᵢ = rowsum(dOᵢ ∘ Oᵢ) = rowsum(Pᵢ ∘ dPᵢ)`. Implement **both** sides and confirm they're equal — this is the identity that lets the backward avoid storing `P`. Fill in the TODO.


In [ ]:
def D_from_outputs(dO, O):
    # TODO: cheap form used by Flash Attention -- D_i = rowsum(dO_i * O_i)
    return np.zeros((O.shape[0], 1))   # replace

def D_from_probs(P, dP):
    # expensive form (needs the N x N matrices): D_i = rowsum(P_i * dP_i)
    return (P * dP).sum(axis=1, keepdims=True)

np.random.seed(3)
Nx, dx = 80, 40
Qx, Kx, Vx = (np.random.randn(Nx, dx) for _ in range(3))
dOx = np.random.randn(Nx, dx)
sc = 1.0 / np.sqrt(dx)
Px = naive_softmax((Qx @ Kx.T) * sc)
Ox = Px @ Vx
dPx = dOx @ Vx.T
e = np.abs(D_from_outputs(dOx, Ox) - D_from_probs(Px, dPx)).max()
print(f"max diff {e:.2e}  ->  {'PASS' if e < 1e-10 else 'FAIL (still the stub)'}")


#### Solution

In [ ]:
def D_from_outputs(dO, O):
    return (dO * O).sum(axis=1, keepdims=True)

e = np.abs(D_from_outputs(dOx, Ox) - D_from_probs(Px, dPx)).max()
assert e < 1e-10
print(f"D_from_outputs == D_from_probs (max diff {e:.2e})  PASS")


## Further Reading

**Source of truth**

- [*FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*](https://arxiv.org/abs/2205.14135) — Algorithm 1 (forward), Appendix B (backward), Theorems 1-2 (FLOPs, IO complexity), Figure 2 (benchmarks).
- [*FlashAttention-2*](https://arxiv.org/abs/2307.08691) — loop reordering, parallelism over sequence length, warp work-partitioning.

**Minimal implementations (read these next)**

- [`tspeterkim/flash-attention-minimal`](https://github.com/tspeterkim/flash-attention-minimal) — ~100-line CUDA forward, the cleanest starting point.
- [`66RING/tiny-flash-attention`](https://github.com/66RING/tiny-flash-attention) — forward **and** backward CUDA kernels.
- [`sonnyli/flash_attention_from_scratch`](https://github.com/sonnyli/flash_attention_from_scratch) — derivations plus a performance-tuned kernel and writeup.

**In this repo**

- [`llmc/cudnn_att.h`](llmc/cudnn_att.h) — the production cuDNN Flash Attention path (Chapter 16).
- [`dev/cuda/attention_forward.cu`](dev/cuda/attention_forward.cu) — multiple attention kernels to compare against.


## Recap

You now know:

- **IO-awareness** is the thesis: attention is bound by HBM traffic, so the win comes from *not* materializing the `(T, T)` scores — even at the cost of extra FLOPs.
- **Online softmax** computes the exact softmax in one pass with `O(1)` state per row, via a running `(m, ℓ)` and a rescaling correction `e^{m−m^new}`.
- **Tiling** turns that into Algorithm 1: stream K/V blocks past each Q block, keeping `(O, m, ℓ)` in SRAM. You wrote and verified this kernel in CUDA.
- **IO complexity** drops from `Θ(N²)` to `Θ(N²d²/M)` HBM accesses — provably near-optimal, ~9x less traffic on GPT-2 medium.
- The **backward** pass recomputes `S, P` block-by-block and uses `Dᵢ = rowsum(dOᵢ ∘ Oᵢ)` to collapse the softmax Jacobian — exact gradients with `O(N)` saved state.
- **FlashAttention-2** rearranges the same algorithm (deferred normalization, Q-outer loop, split-Q across warps) for ~2x more throughput.

### What's next

Back to the main track: **Chapter 17 — Mixed Precision (BF16/FP16).** Flash Attention in production runs in bfloat16 on tensor cores; Chapter 17 is the precision machinery (`floatX`, master weights) that makes that stable. When you're ready, say **"proceed to Chapter 17"**.
